# Turkey City District Code Table (Central Primary Key)

Builds the single reference table `city_code | city_name | district_code | district_name` from the two TÜİK MEDAS exports
`il_nufus_verisi.csv` (`Adana-1`) and `ilce_nufus_verisi.csv` (`Adana(Aladağ)-1757`).
Every other district-level table can be joined on `district_code` (or `city_code` for provinces) to get consistent names and codes.
The table reflects the 2025 administrative structure; it covers every district that exists from 2013 onward.
Input and output live in the same folder (`data/raw/`); `district_name` is **not** unique (e.g. 51 × "Merkez"), only `district_code` is.

In [ ]:
# Central reference table: city_code | city_name | district_code | district_name  (join key for every district-level table)
import os
import pandas as pd

DATA_DIR = "../data/raw"  # inputs and output share this folder
CITY_PATH = os.path.join(DATA_DIR, "il_nufus_verisi.csv")
DISTRICT_PATH = os.path.join(DATA_DIR, "ilce_nufus_verisi.csv")
CODE_TABLE_PATH = os.path.join(DATA_DIR, "turkey_city_district_code_table.csv")

# 'Adana-1'  /  'Adana(Aladağ)-1757'  ->  name(s) + code
CITY_LABEL_PATTERN = r'^(?P<city_name>.*)-(?P<city_code>\d+)$'
DISTRICT_LABEL_PATTERN = r'^(?P<city_name>.*?)\((?P<district_name>.*)\)-(?P<district_code>\d+)$'


def _parse_labels(path: str, pattern: str) -> pd.DataFrame:
    """MEDAS pivot export -> one row per 'Name-Code' label; header/separator rows are NaN and dropped."""
    labels = pd.read_csv(
        path, sep='|', header=None, usecols=[1], names=['label'], encoding='utf-8-sig',
    )['label'].dropna().str.strip()
    parsed = labels.str.extract(pattern)
    if parsed.isna().any(axis=None):
        raise ValueError(f"Unparseable labels in {path}: {labels[parsed.isna().any(axis=1)].tolist()}")
    return parsed


def build_district_code_table(city_path: str, district_path: str) -> pd.DataFrame:
    cities = _parse_labels(city_path, CITY_LABEL_PATTERN)
    districts = _parse_labels(district_path, DISTRICT_LABEL_PATTERN)

    # The district labels carry the city *name*; the city code comes from the city file.
    table = districts.merge(cities, on='city_name', how='left', validate='many_to_one')
    if table['city_code'].isna().any():
        raise ValueError(f"Cities missing from {city_path}: {sorted(table.loc[table['city_code'].isna(), 'city_name'].unique())}")
    if table['district_code'].duplicated().any():
        raise ValueError(f"Duplicate district_code: {sorted(table.loc[table['district_code'].duplicated(), 'district_code'].unique())}")

    return (
        table.astype({'city_code': int, 'district_code': int})[['city_code', 'city_name', 'district_code', 'district_name']]
        .sort_values(['city_code', 'district_code'], ignore_index=True)
    )


df_codes = build_district_code_table(CITY_PATH, DISTRICT_PATH)
df_codes.to_csv(CODE_TABLE_PATH, index=False)
print(f"{len(df_codes)} districts across {df_codes['city_code'].nunique()} cities -> {CODE_TABLE_PATH}")
df_codes.head(5)
